# CNN+LSTM DDoS Detector — Offline Training Notebook

**End-to-end pipeline**: Data audit → Feature engineering → Leakage prevention → CNN+LSTM training → Optuna tuning → Evaluation → Calibration → CV stability → PPO-ready output.

All plots are saved as **separate PNG files** in `outputs_detector/`.

## 0. Configuration & Imports

In [1]:
# ═══════════════════════════════════════════════════════════════════
# Section 0 — Configuration & Imports
# ═══════════════════════════════════════════════════════════════════
import os, sys, json, pickle, re, warnings, hashlib, time
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")          # headless-friendly; switch to inline for notebooks
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, GroupShuffleSplit
)
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, confusion_matrix,
    classification_report, precision_recall_curve, roc_curve,
    brier_score_loss
)
from sklearn.impute import SimpleImputer

import tensorflow as tf
from tensorflow import keras
from keras import layers, regularizers, callbacks, backend as K

import optuna
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# ── Tunables ────────────────────────────────────────────────────────
DATA_PATH       = "CNN+LSTM_dataset_FINAL_merged_sorted.csv"
OUTPUT_DIR      = "outputs_detector"
LABEL_COL       = "label"
SEED            = 42
WINDOW_LEN      = 10          # default; Optuna will tune
STRIDE          = 1           # default; Optuna will tune
TEST_FRAC       = 0.15
VAL_FRAC        = 0.15
OPTUNA_TRIALS   = 30
CV_FOLDS        = 5
BATCH_SIZE      = 64
MAX_EPOCHS      = 100
EARLY_STOP_PAT  = 10
REDUCE_LR_PAT   = 5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Reproducibility ────────────────────────────────────────────────
def seed_everything(seed=SEED):
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["TF_DETERMINISTIC_OPS"] = "1"

seed_everything()

# ── Helper: save figure ────────────────────────────────────────────
def save_fig(fig, name, dpi=200):
    path = os.path.join(OUTPUT_DIR, f"{name}.png")
    fig.savefig(path, dpi=dpi, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  ✅ Saved plot: {path}")
    return path

print(f"TensorFlow {tf.__version__}  |  GPU: {tf.config.list_physical_devices('GPU')}")
print(f"Output dir: {os.path.abspath(OUTPUT_DIR)}")


2026-02-18 14:22:44.884912: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TensorFlow 2.20.0  |  GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Output dir: /home/shokran/PycharmProjects/JupyterProject/Models/newTrail/outputs_detector


/home/shokran/PycharmProjects/JupyterProject/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Audit & Validation

In [2]:
# ═══════════════════════════════════════════════════════════════════
# Section 1 — Data Audit & Validation
# ═══════════════════════════════════════════════════════════════════
raw = pd.read_csv(DATA_PATH)
print(f"Shape: {raw.shape}")
print(f"Columns ({len(raw.columns)}):")
for i, c in enumerate(raw.columns):
    print(f"  {i:3d}  {c:40s}  {str(raw[c].dtype):10s}")


Shape: (9111, 162)
Columns (162):
    0  schema                                    object    
    1  timestamp_utc                             object    
    2  timestamp_unix                            float64   
    3  t_monotonic                               float64   
    4  sample_id                                 int64     
    5  label                                     int64     
    6  onos1_flows                               int64     
    7  onos1_packets                             int64     
    8  onos1_bytes                               int64     
    9  onos1_flows_rate                          float64   
   10  onos1_packets_rate                        float64   
   11  onos1_bytes_rate                          float64   
   12  onos1_cpu                                 float64   
   13  onos1_mem                                 object    
   14  onos1_pktin_total                         float64   
   15  onos1_pktin_total_rate                    float64   
   16 

In [3]:
# ── 1a. Missing values summary ─────────────────────────────────────
missing = raw.isnull().sum()
missing_pct = (missing / len(raw)) * 100
miss_df = pd.DataFrame({"count": missing, "pct": missing_pct})
miss_df = miss_df[miss_df["count"] > 0].sort_values("pct", ascending=False)
print("\n── Missing Values ──")
print(miss_df.to_string())

fig, ax = plt.subplots(figsize=(10, max(4, len(miss_df)*0.35)))
miss_df["pct"].plot.barh(ax=ax, color="salmon")
ax.set_xlabel("Missing %")
ax.set_title("Missing Values per Column")
ax.invert_yaxis()
save_fig(fig, "01_missing_values_bar")



── Missing Values ──
                       count         pct
onos1_event_queue       9111  100.000000
onos1_gc_pause_ms       9111  100.000000
onos2_event_queue       9111  100.000000
onos2_gc_pause_ms       9111  100.000000
onos3_event_queue       9111  100.000000
onos3_gc_pause_ms       9111  100.000000
warmup                  6666   73.164307
well_known_port_flows   2445   26.835693
syn_flows               2445   26.835693
syn_ratio               2445   26.835693
five_unique             2445   26.835693
  ✅ Saved plot: outputs_detector/01_missing_values_bar.png


'outputs_detector/01_missing_values_bar.png'

In [4]:
# ── 1b. Duplicates ─────────────────────────────────────────────────
n_dup = raw.duplicated().sum()
print(f"\nDuplicate rows: {n_dup}")
if n_dup > 0:
    raw = raw.drop_duplicates().reset_index(drop=True)
    print(f"  → Dropped duplicates. New shape: {raw.shape}")



Duplicate rows: 0


In [5]:
# ── 1c. Constant / near-constant columns ───────────────────────────
nuniq = raw.nunique()
const_cols = nuniq[nuniq <= 1].index.tolist()
print(f"\nConstant columns (nunique ≤ 1): {const_cols}")



Constant columns (nunique ≤ 1): ['onos1_event_queue', 'onos1_atomix_nodes', 'onos1_gc_pause_ms', 'onos2_event_queue', 'onos2_atomix_nodes', 'onos2_gc_pause_ms', 'onos3_event_queue', 'onos3_atomix_nodes', 'onos3_gc_pause_ms', 's1_master', 's2_master', 's3_master', 's4_master', 'cluster_size', 'active_controllers', 'cfg_interval_s', 'cfg_switches', 'cfg_controllers', 'warmup']


In [6]:
# ── 1d. GiB/MiB/KiB string conversion ──────────────────────────────
_UNIT_MAP = {"gib": 2**30, "mib": 2**20, "kib": 2**10,
             "gb": 1e9, "mb": 1e6, "kb": 1e3, "b": 1}

def parse_mem(val):
    if pd.isna(val):
        return np.nan
    if isinstance(val, (int, float)):
        return float(val)
    s = str(val).strip().lower()
    for suffix, mult in sorted(_UNIT_MAP.items(), key=lambda x: -len(x[0])):
        if s.endswith(suffix):
            try:
                return float(s[: -len(suffix)]) * mult
            except ValueError:
                return np.nan
    try:
        return float(s)
    except ValueError:
        return np.nan

mem_cols = [c for c in raw.columns if raw[c].dtype == "object"
            and raw[c].dropna().astype(str).str.contains(
                r"[0-9].*[GgMmKk]i?[Bb]", regex=True).any()]
print(f"\nMemory-string columns detected: {mem_cols}")
for c in mem_cols:
    raw[c] = raw[c].apply(parse_mem)
    print(f"  Converted {c} → numeric bytes  (sample: {raw[c].iloc[0]:.0f})")



Memory-string columns detected: ['onos1_mem', 'onos2_mem', 'onos3_mem']
  Converted onos1_mem → numeric bytes  (sample: 1758789108)
  Converted onos2_mem → numeric bytes  (sample: 1217623228)
  Converted onos3_mem → numeric bytes  (sample: 1330366120)


In [7]:
# ── 1e. Inf / -Inf check ───────────────────────────────────────────
num_cols = raw.select_dtypes(include=[np.number]).columns
inf_counts = np.isinf(raw[num_cols]).sum()
inf_cols = inf_counts[inf_counts > 0]
print(f"\nColumns with inf values: {len(inf_cols)}")
if len(inf_cols):
    print(inf_cols)
    raw[num_cols] = raw[num_cols].replace([np.inf, -np.inf], np.nan)
    print("  → Replaced inf with NaN")



Columns with inf values: 0


In [8]:
# ── 1f. Label distribution ─────────────────────────────────────────
label_counts = raw[LABEL_COL].value_counts().sort_index()
print(f"\n── Label Distribution ──")
print(label_counts)
imbalance_ratio = label_counts.min() / label_counts.max()
print(f"Minority/Majority ratio: {imbalance_ratio:.3f}")

fig, ax = plt.subplots(figsize=(5, 4))
label_counts.plot.bar(ax=ax, color=["steelblue", "firebrick"],
                       edgecolor="black")
ax.set_title("Label Distribution")
ax.set_xlabel(LABEL_COL); ax.set_ylabel("Count")
for i, v in enumerate(label_counts):
    ax.text(i, v + 30, str(v), ha="center", fontweight="bold")
save_fig(fig, "01_label_distribution")



── Label Distribution ──
label
0    5112
1    3999
Name: count, dtype: int64
Minority/Majority ratio: 0.782
  ✅ Saved plot: outputs_detector/01_label_distribution.png


'outputs_detector/01_label_distribution.png'

In [9]:
# ── 1g. Timestamp ordering check ───────────────────────────────────
TS_COL = None
for candidate in ["timestamp_unix", "timestamp_utc", "t_monotonic"]:
    if candidate in raw.columns:
        TS_COL = candidate; break

if TS_COL:
    ts = pd.to_numeric(raw[TS_COL], errors="coerce")
    is_sorted = ts.is_monotonic_increasing
    print(f"\nTimestamp column: {TS_COL}")
    print(f"  Sorted ascending: {is_sorted}")
    if not is_sorted:
        raw = raw.sort_values(TS_COL).reset_index(drop=True)
        print("  → Re-sorted by timestamp.")
else:
    print("\n⚠ No timestamp column found.")

print(f"\n═══ Data Audit complete. Shape after cleaning: {raw.shape} ═══")



Timestamp column: timestamp_unix
  Sorted ascending: False
  → Re-sorted by timestamp.

═══ Data Audit complete. Shape after cleaning: (9111, 162) ═══


## 2. Feature Mapping & Calculations

In [10]:
# ═══════════════════════════════════════════════════════════════════
# Section 2 — Feature Mapping & Calculations
# ═══════════════════════════════════════════════════════════════════
df = raw.copy()
mapping_log = []

def safe_map(src, tgt, frame):
    """Rename src→tgt. If both exist, keep src (cluster version) and drop tgt."""
    if src in frame.columns and tgt in frame.columns:
        frame.drop(columns=[tgt], inplace=True)
        frame.rename(columns={src: tgt}, inplace=True)
        mapping_log.append(f"✅ Both {src} & {tgt} exist → kept {src}, renamed to {tgt}")
    elif src in frame.columns:
        frame.rename(columns={src: tgt}, inplace=True)
        mapping_log.append(f"✅ Mapped: {src} → {tgt}")
    else:
        mapping_log.append(f"ℹ️  {src} not found; {tgt} already present or N/A")

safe_map("cluster_syn_flows", "syn_flows", df)
safe_map("cluster_syn_ratio", "syn_ratio", df)

# ── well_known_port_flows (proxy) ──────────────────────────────────
# Since raw port lists are unavailable, we use a proxy:
#   ratio of well-known range capacity to observed unique dst ports.
if "unique_dst_ports" in df.columns:
    udp = df["unique_dst_ports"].fillna(0).clip(lower=0)
    if "well_known_port_flows" not in df.columns or df["well_known_port_flows"].isna().all():
        df["well_known_port_flows"] = np.where(
            udp > 0,
            np.minimum(udp, 1024) / np.maximum(udp, 1),
            0.0
        )
        mapping_log.append(
            "✅ Calculated: well_known_port_flows = min(unique_dst_ports, 1024) "
            "/ max(unique_dst_ports, 1)  [proxy — raw port lists unavailable]"
        )
    else:
        # Fill missing values with the proxy
        mask = df["well_known_port_flows"].isna()
        df.loc[mask, "well_known_port_flows"] = np.where(
            udp[mask] > 0,
            np.minimum(udp[mask], 1024) / np.maximum(udp[mask], 1),
            0.0
        )
        mapping_log.append(
            "✅ Filled missing well_known_port_flows with proxy formula"
        )

# ── five_unique ────────────────────────────────────────────────────
if "unique_srcs" in df.columns and "unique_dsts" in df.columns:
    if "five_unique" not in df.columns or df["five_unique"].isna().all():
        df["five_unique"] = df["unique_srcs"].fillna(0) + df["unique_dsts"].fillna(0)
        mapping_log.append("✅ Calculated: five_unique = unique_srcs + unique_dsts")
    else:
        mask = df["five_unique"].isna()
        df.loc[mask, "five_unique"] = (
            df.loc[mask, "unique_srcs"].fillna(0) + df.loc[mask, "unique_dsts"].fillna(0)
        )
        mapping_log.append("✅ Filled missing five_unique from unique_srcs + unique_dsts")
else:
    mapping_log.append("⚠ unique_srcs/unique_dsts not found; five_unique not computed")

print("\n🔧 FEATURE MAPPING & CALCULATION")
for line in mapping_log:
    print(f"  {line}")


🔧 FEATURE MAPPING & CALCULATION
  ✅ Both cluster_syn_flows & syn_flows exist → kept cluster_syn_flows, renamed to syn_flows
  ✅ Both cluster_syn_ratio & syn_ratio exist → kept cluster_syn_ratio, renamed to syn_ratio
  ✅ Filled missing well_known_port_flows with proxy formula
  ✅ Filled missing five_unique from unique_srcs + unique_dsts


## 3. Leakage-Safe Feature Filtering

In [11]:
# ═══════════════════════════════════════════════════════════════════
# Section 3 — Leakage-Safe Feature Filtering
# ═══════════════════════════════════════════════════════════════════

# Columns to DROP with reasons
drop_reasons = {}

# 3a. Timestamp / time columns (kept aside for chrono split only)
time_cols = ["timestamp_utc", "timestamp_unix", "t_monotonic"]
for c in time_cols:
    if c in df.columns:
        drop_reasons[c] = "timestamp / time column"

# 3b. IDs, run numbers, step indexes
id_cols = ["sample_id", "schema"]
for c in id_cols:
    if c in df.columns:
        drop_reasons[c] = "ID / schema column"

# 3c. IP addresses, topology strings, controller names, filenames
leak_cols = ["top_dst", "cfg_switches", "cfg_controllers",
             "source_file", "warmup",
             "s1_master", "s2_master", "s3_master", "s4_master"]
for c in leak_cols:
    if c in df.columns:
        drop_reasons[c] = "IP / topology / filename — leakage risk"

# 3d. Label aliases / future info
alias_cols = ["attack_in_progress"]
for c in alias_cols:
    if c in df.columns:
        drop_reasons[c] = "label alias — leakage"

# 3e. Config columns that are constant or near-constant
config_cols = ["cfg_interval_s", "cluster_size", "active_controllers",
               "tracked_flows"]
for c in config_cols:
    if c in df.columns:
        if df[c].nunique() <= 2:
            drop_reasons[c] = "config / near-constant"

# 3f. 100% missing columns
for c in df.columns:
    if df[c].isna().all():
        drop_reasons[c] = "100% missing"

# Keep source_file aside for group splitting before dropping
source_file_series = df["source_file"].copy() if "source_file" in df.columns else None
# Keep timestamp for chrono split
ts_for_split = df["timestamp_unix"].copy() if "timestamp_unix" in df.columns else None

# Perform drops
to_drop = [c for c in drop_reasons if c in df.columns]
df.drop(columns=to_drop, inplace=True)

print("── Dropped Columns ──")
for c, reason in sorted(drop_reasons.items()):
    print(f"  ✗ {c:35s} → {reason}")
print(f"\nRemaining columns: {len(df.columns)}")


── Dropped Columns ──
  ✗ active_controllers                  → config / near-constant
  ✗ attack_in_progress                  → label alias — leakage
  ✗ cfg_controllers                     → IP / topology / filename — leakage risk
  ✗ cfg_interval_s                      → config / near-constant
  ✗ cfg_switches                        → IP / topology / filename — leakage risk
  ✗ cluster_size                        → config / near-constant
  ✗ onos1_event_queue                   → 100% missing
  ✗ onos1_gc_pause_ms                   → 100% missing
  ✗ onos2_event_queue                   → 100% missing
  ✗ onos2_gc_pause_ms                   → 100% missing
  ✗ onos3_event_queue                   → 100% missing
  ✗ onos3_gc_pause_ms                   → 100% missing
  ✗ s1_master                           → IP / topology / filename — leakage risk
  ✗ s2_master                           → IP / topology / filename — leakage risk
  ✗ s3_master                           → IP / topology / fil

In [12]:
# ── 3g. Correlation scan with label ─────────────────────────────────
numeric_df = df.select_dtypes(include=[np.number])
corr_with_label = numeric_df.corr()[LABEL_COL].drop(LABEL_COL).abs().sort_values(ascending=False)
print("\n── Top-15 features correlated with label (|r|) ──")
print(corr_with_label.head(15).to_string())

suspicious = corr_with_label[corr_with_label > 0.95]
if len(suspicious):
    print(f"\n⚠ SUSPICIOUS (|r| > 0.95): {suspicious.index.tolist()}")
    print("  Consider dropping these to avoid label leakage!")
else:
    print("\n✅ No features with |r| > 0.95 with label.")

# ── Correlation heatmap (top 30) ───────────────────────────────────
top30 = corr_with_label.head(30).index.tolist() + [LABEL_COL]
corr_matrix = numeric_df[top30].corr()
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(corr_matrix, annot=False, cmap="coolwarm", center=0,
            linewidths=0.3, ax=ax)
ax.set_title("Correlation Heatmap — Top 30 Features + Label")
save_fig(fig, "03_correlation_heatmap")



── Top-15 features correlated with label (|r|) ──
onos2_other_flows          0.944236
onos3_other_flows          0.940845
onos1_other_flows          0.935159
cluster_other_flows        0.691588
icmp_flows                 0.627033
tcp_flows                  0.525192
onos3_rest_flows_max_ms    0.508456
avg_pkt_size               0.508203
onos2_icmp_flows           0.507729
cluster_icmp_flows         0.507342
onos3_icmp_flows           0.506037
onos1_icmp_flows           0.501962
port_diversity             0.489108
onos2_rest_flows_max_ms    0.486360
onos1_rest_hosts_max_ms    0.481203

✅ No features with |r| > 0.95 with label.
  ✅ Saved plot: outputs_detector/03_correlation_heatmap.png


'outputs_detector/03_correlation_heatmap.png'

In [13]:
# ── Final feature list ──────────────────────────────────────────────
feature_cols = [c for c in df.columns if c != LABEL_COL
                and df[c].dtype in [np.float64, np.float32, np.int64, np.int32, float, int]]
# Drop any remaining object columns
obj_remaining = df.select_dtypes(include=["object"]).columns.tolist()
if obj_remaining:
    print(f"Dropping remaining object columns: {obj_remaining}")
    df.drop(columns=obj_remaining, inplace=True)
    feature_cols = [c for c in feature_cols if c not in obj_remaining]

print(f"\n═══ Final feature set: {len(feature_cols)} features ═══")
for i, c in enumerate(feature_cols):
    print(f"  {i:3d}  {c}")



═══ Final feature set: 135 features ═══
    0  onos1_flows
    1  onos1_packets
    2  onos1_bytes
    3  onos1_flows_rate
    4  onos1_packets_rate
    5  onos1_bytes_rate
    6  onos1_cpu
    7  onos1_mem
    8  onos1_pktin_total
    9  onos1_pktin_total_rate
   10  onos1_pktout_total
   11  onos1_pktout_total_rate
   12  onos1_flowmiss_total
   13  onos1_flowmiss_total_rate
   14  onos1_atomix_nodes
   15  onos1_rest_flows_avg_ms
   16  onos1_rest_flows_max_ms
   17  onos1_rest_hosts_avg_ms
   18  onos1_rest_hosts_max_ms
   19  onos1_rest_metrics_avg_ms
   20  onos1_rest_metrics_max_ms
   21  onos1_tcp_flows
   22  onos1_udp_flows
   23  onos1_icmp_flows
   24  onos1_other_flows
   25  onos1_syn_flows
   26  onos2_flows
   27  onos2_packets
   28  onos2_bytes
   29  onos2_flows_rate
   30  onos2_packets_rate
   31  onos2_bytes_rate
   32  onos2_cpu
   33  onos2_mem
   34  onos2_pktin_total
   35  onos2_pktin_total_rate
   36  onos2_pktout_total
   37  onos2_pktout_total_rate
   38 

## 4. Splitting Strategy

Two evaluation modes: **A) Stratified random** and **B) Chronological**.

In [14]:
# ═══════════════════════════════════════════════════════════════════
# Section 4 — Splitting Strategy
# ═══════════════════════════════════════════════════════════════════
X_all = df[feature_cols].values
y_all = df[LABEL_COL].values

# ── Mode A: Stratified random split ────────────────────────────────
idx = np.arange(len(y_all))
idx_train_val, idx_test_A, y_tv, y_te = train_test_split(
    idx, y_all, test_size=TEST_FRAC, stratify=y_all, random_state=SEED)
relative_val = VAL_FRAC / (1 - TEST_FRAC)
idx_train_A, idx_val_A, _, _ = train_test_split(
    idx_train_val, y_tv, test_size=relative_val, stratify=y_tv, random_state=SEED)

print("── Mode A: Stratified Random Split ──")
for name, idxs in [("Train", idx_train_A), ("Val", idx_val_A), ("Test", idx_test_A)]:
    labs = y_all[idxs]
    print(f"  {name:5s}: n={len(idxs):5d}  |  "
          f"class 0: {(labs==0).sum():4d} ({(labs==0).mean():.1%})  |  "
          f"class 1: {(labs==1).sum():4d} ({(labs==1).mean():.1%})")

# ── Mode B: Chronological split ────────────────────────────────────
if ts_for_split is not None:
    chrono_order = np.argsort(ts_for_split.values)
    n = len(chrono_order)
    n_train = int(n * (1 - TEST_FRAC - VAL_FRAC))
    n_val = int(n * VAL_FRAC)
    idx_train_B = chrono_order[:n_train]
    idx_val_B   = chrono_order[n_train:n_train + n_val]
    idx_test_B  = chrono_order[n_train + n_val:]
    print("\n── Mode B: Chronological Split ──")
    for name, idxs in [("Train", idx_train_B), ("Val", idx_val_B), ("Test", idx_test_B)]:
        labs = y_all[idxs]
        print(f"  {name:5s}: n={len(idxs):5d}  |  "
              f"class 0: {(labs==0).sum():4d} ({(labs==0).mean():.1%})  |  "
              f"class 1: {(labs==1).sum():4d} ({(labs==1).mean():.1%})")
    HAS_CHRONO = True
else:
    print("\n⚠ No timestamp found — skipping chronological split.")
    HAS_CHRONO = False
    idx_train_B = idx_val_B = idx_test_B = None

# Save split indices
np.savez(os.path.join(OUTPUT_DIR, "split_indices.npz"),
         train_A=idx_train_A, val_A=idx_val_A, test_A=idx_test_A,
         train_B=idx_train_B if HAS_CHRONO else np.array([]),
         val_B=idx_val_B if HAS_CHRONO else np.array([]),
         test_B=idx_test_B if HAS_CHRONO else np.array([]))
print(f"\n✅ Split indices saved to {OUTPUT_DIR}/split_indices.npz")


── Mode A: Stratified Random Split ──
  Train: n= 6377  |  class 0: 3578 (56.1%)  |  class 1: 2799 (43.9%)
  Val  : n= 1367  |  class 0:  767 (56.1%)  |  class 1:  600 (43.9%)
  Test : n= 1367  |  class 0:  767 (56.1%)  |  class 1:  600 (43.9%)

── Mode B: Chronological Split ──
  Train: n= 6377  |  class 0: 4111 (64.5%)  |  class 1: 2266 (35.5%)
  Val  : n= 1366  |  class 0:  418 (30.6%)  |  class 1:  948 (69.4%)
  Test : n= 1368  |  class 0:  583 (42.6%)  |  class 1:  785 (57.4%)

✅ Split indices saved to outputs_detector/split_indices.npz


## 5. Preprocessing Pipeline

Median imputation → StandardScaler (fit on train only) → Sequence windowing.

In [15]:
# ═══════════════════════════════════════════════════════════════════
# Section 5 — Preprocessing Pipeline
# ═══════════════════════════════════════════════════════════════════

def preprocess_split(X_all, y_all, idx_tr, idx_va, idx_te, feature_cols,
                     window_len=WINDOW_LEN, stride=STRIDE, fit_new=True,
                     imputer=None, scaler=None):
    """Impute, scale, window. Returns arrays + fitted imputer/scaler."""
    X_tr, X_va, X_te = X_all[idx_tr], X_all[idx_va], X_all[idx_te]
    y_tr, y_va, y_te = y_all[idx_tr], y_all[idx_va], y_all[idx_te]

    # Impute
    if fit_new or imputer is None:
        imputer = SimpleImputer(strategy="median")
        X_tr = imputer.fit_transform(X_tr)
    else:
        X_tr = imputer.transform(X_tr)
    X_va = imputer.transform(X_va)
    X_te = imputer.transform(X_te)

    # Scale
    if fit_new or scaler is None:
        scaler = StandardScaler()
        X_tr = scaler.fit_transform(X_tr)
    else:
        X_tr = scaler.transform(X_tr)
    X_va = scaler.transform(X_va)
    X_te = scaler.transform(X_te)

    # Window
    Xw_tr, yw_tr = create_sequences(X_tr, y_tr, window_len, stride)
    Xw_va, yw_va = create_sequences(X_va, y_va, window_len, stride)
    Xw_te, yw_te = create_sequences(X_te, y_te, window_len, stride)

    return (Xw_tr, yw_tr, Xw_va, yw_va, Xw_te, yw_te), imputer, scaler


def create_sequences(X, y, window_len, stride=1):
    """Slide a window over X, y. Label = last timestep in window."""
    Xs, ys = [], []
    for i in range(0, len(X) - window_len + 1, stride):
        Xs.append(X[i : i + window_len])
        ys.append(y[i + window_len - 1])      # label at END of window
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)

# ── Preprocess Mode-A split ─────────────────────────────────────────
(Xw_tr_A, yw_tr_A, Xw_va_A, yw_va_A, Xw_te_A, yw_te_A), \
    main_imputer, main_scaler = preprocess_split(
        X_all, y_all, idx_train_A, idx_val_A, idx_test_A, feature_cols)

print(f"Windowed shapes (Mode A, window={WINDOW_LEN}, stride={STRIDE}):")
print(f"  Train:  X={Xw_tr_A.shape}  y={yw_tr_A.shape}")
print(f"  Val:    X={Xw_va_A.shape}  y={yw_va_A.shape}")
print(f"  Test:   X={Xw_te_A.shape}  y={yw_te_A.shape}")

# Save scaler
with open(os.path.join(OUTPUT_DIR, "scaler.pkl"), "wb") as f:
    pickle.dump(main_scaler, f)
with open(os.path.join(OUTPUT_DIR, "imputer.pkl"), "wb") as f:
    pickle.dump(main_imputer, f)
print(f"✅ Scaler and imputer saved.")


Windowed shapes (Mode A, window=10, stride=1):
  Train:  X=(6368, 10, 135)  y=(6368,)
  Val:    X=(1358, 10, 135)  y=(1358,)
  Test:   X=(1358, 10, 135)  y=(1358,)
✅ Scaler and imputer saved.


## 6. CNN+LSTM Model + Anti-Overfit Controls

In [16]:
# ═══════════════════════════════════════════════════════════════════
# Section 6 — Model Builder + Training Helpers
# ═══════════════════════════════════════════════════════════════════

def build_cnn_lstm(window_len, n_features,
                   cnn_filters=64, kernel_size=3, n_cnn_layers=2,
                   lstm_units=64, dropout_rate=0.4,
                   l2_reg=1e-3, learning_rate=1e-3):
    """Build a CNN+LSTM binary classifier with strong regularisation."""
    inp = layers.Input(shape=(window_len, n_features))
    x = inp

    # CNN blocks
    for i in range(n_cnn_layers):
        x = layers.Conv1D(cnn_filters, kernel_size, padding="same",
                          kernel_regularizer=regularizers.l2(l2_reg))(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
        x = layers.SpatialDropout1D(dropout_rate * 0.5)(x)   # dropout INSIDE CNN

    # LSTM
    x = layers.LSTM(lstm_units, return_sequences=False,
                    kernel_regularizer=regularizers.l2(l2_reg),
                    recurrent_regularizer=regularizers.l2(l2_reg),
                    recurrent_dropout=0.2)(x)
    x = layers.Dropout(dropout_rate)(x)

    # Dense head (smaller)
    x = layers.Dense(32, activation="relu",
                     kernel_regularizer=regularizers.l2(l2_reg))(x)
    x = layers.Dropout(dropout_rate)(x)
    out = layers.Dense(1, activation="sigmoid")(x)

    model = keras.Model(inp, out)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate,
                                       clipnorm=1.0),
        loss="binary_crossentropy",
        metrics=[keras.metrics.AUC(name="auc", curve="ROC"),
                 keras.metrics.AUC(name="pr_auc", curve="PR")]
    )
    return model


def compute_class_weights(y):
    from sklearn.utils.class_weight import compute_class_weight
    cw = compute_class_weight("balanced", classes=np.unique(y), y=y)
    return dict(enumerate(cw))


def get_callbacks(monitor="val_pr_auc", mode="max"):
    return [
        callbacks.EarlyStopping(
            monitor=monitor, mode=mode,
            patience=5, restore_best_weights=True, verbose=1),
        callbacks.ReduceLROnPlateau(
            monitor=monitor, mode=mode,
            patience=3, factor=0.5, min_lr=1e-6, verbose=1),
    ]


def train_and_evaluate(model, Xw_tr, yw_tr, Xw_va, yw_va,
                       batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
                       class_weight=None, verbose=1):
    """Train model, return history."""
    if class_weight is None:
        class_weight = compute_class_weights(yw_tr)
    history = model.fit(
        Xw_tr, yw_tr,
        validation_data=(Xw_va, yw_va),
        batch_size=batch_size,
        epochs=epochs,
        class_weight=class_weight,
        callbacks=get_callbacks(),
        verbose=verbose
    )
    return history

print("✅ Model builder and training helpers defined.")

✅ Model builder and training helpers defined.


### 6.1 Baseline Model — Quick Sanity Check

In [17]:
# ── Baseline training (Mode A, default hypers) ─────────────────────
n_features = Xw_tr_A.shape[2]
baseline_model = build_cnn_lstm(WINDOW_LEN, n_features)
baseline_model.summary()

baseline_history = train_and_evaluate(
    baseline_model, Xw_tr_A, yw_tr_A, Xw_va_A, yw_va_A, verbose=0)


I0000 00:00:1771406613.626382   65614 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 2141 MB memory:  -> device: 0, name: NVIDIA GeForce GTX 1650, pci bus id: 0000:01:00.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 10, 135)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 10, 64)         │        25,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d               │ (None, 10, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 10, 64)         │        12,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 10, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 10, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ spatial_dropout1d_1             │ (None, 10, 64)         │             0 │
│ (SpatialDropout1D)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 73,985 (289.00 KB)

 Trainable params: 73,729 (288.00 KB)

 Non-trainable params: 256 (1.00 KB)

2026-02-18 14:23:34.843760: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-02-18 14:23:39.321676: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:473] Loaded cuDNN version 91002
2026-02-18 14:23:45.881121: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=Map


Epoch 6: ReduceLROnPlateau reducing learning rate to 0.0005000000237487257.
Epoch 8: early stopping
Restoring model weights from the end of the best epoch: 3.


### 6.2 Model Diagnosis — Overfit / Underfit Detection

In [18]:
# ── Training curves ─────────────────────────────────────────────────
def plot_training_curves(history, prefix="06"):
    h = history.history

    # Loss
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(h["loss"], label="Train Loss")
    ax.plot(h["val_loss"], label="Val Loss")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
    ax.set_title("Train vs Val Loss")
    ax.legend(); ax.grid(True, alpha=0.3)
    save_fig(fig, f"{prefix}_loss_curves")

    # PR-AUC
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(h.get("pr_auc", []), label="Train PR-AUC")
    ax.plot(h.get("val_pr_auc", []), label="Val PR-AUC")
    ax.set_xlabel("Epoch"); ax.set_ylabel("PR-AUC")
    ax.set_title("Train vs Val PR-AUC")
    ax.legend(); ax.grid(True, alpha=0.3)
    save_fig(fig, f"{prefix}_pr_auc_curves")

    # Diagnosis text
    final_tr_loss = h["loss"][-1]
    final_va_loss = h["val_loss"][-1]
    gap = final_va_loss - final_tr_loss
    print(f"\n── Model Diagnosis ──")
    print(f"  Final train loss: {final_tr_loss:.4f}")
    print(f"  Final val   loss: {final_va_loss:.4f}")
    print(f"  Gap (val - train): {gap:.4f}")
    if gap > 0.15:
        print("  ⚠ OVERFITTING likely — val loss significantly higher than train.")
    elif final_tr_loss > 0.4:
        print("  ⚠ UNDERFITTING likely — train loss still high.")
    else:
        print("  ✅ Reasonable fit — gap manageable.")

plot_training_curves(baseline_history)


  ✅ Saved plot: outputs_detector/06_loss_curves.png
  ✅ Saved plot: outputs_detector/06_pr_auc_curves.png

── Model Diagnosis ──
  Final train loss: 0.1163
  Final val   loss: 0.1174
  Gap (val - train): 0.0011
  ✅ Reasonable fit — gap manageable.


## 7. Optuna Hyperparameter Optimization

In [19]:
# ═══════════════════════════════════════════════════════════════════
# Section 7 — Optuna Hyperparameter Optimisation
# ═══════════════════════════════════════════════════════════════════

def optuna_objective(trial):
    seed_everything()

    # -- Hyperparameters to tune --
    wl          = trial.suggest_categorical("window_len", [5, 10, 20, 30])
    cnn_filters = trial.suggest_int("cnn_filters", 32, 128, step=16)
    kernel_size = trial.suggest_int("kernel_size", 3, 7, step=2)
    lstm_units  = trial.suggest_int("lstm_units", 32, 128, step=16)
    dropout     = trial.suggest_float("dropout", 0.1, 0.5, step=0.05)
    lr          = trial.suggest_float("lr", 1e-5, 1e-2, log=True)
    batch_size  = trial.suggest_categorical("batch_size", [32, 64, 128])
    l2_reg      = trial.suggest_float("l2_reg", 1e-6, 1e-3, log=True)

    # Re-window with trial window_len
    (Xw_tr, yw_tr, Xw_va, yw_va, _, _), _, _ = preprocess_split(
        X_all, y_all, idx_train_A, idx_val_A, idx_test_A, feature_cols,
        window_len=wl, stride=STRIDE,
        fit_new=False, imputer=main_imputer, scaler=main_scaler)

    if len(Xw_tr) == 0 or len(Xw_va) == 0:
        return 0.0

    nf = Xw_tr.shape[2]
    model = build_cnn_lstm(wl, nf,
                           cnn_filters=cnn_filters,
                           kernel_size=kernel_size,
                           lstm_units=lstm_units,
                           dropout_rate=dropout,
                           l2_reg=l2_reg,
                           learning_rate=lr)

    cw = compute_class_weights(yw_tr)

    # Pruning callback
    class OptunaCallback(keras.callbacks.Callback):
        def on_epoch_end(self, epoch, logs=None):
            val_pr = logs.get("val_pr_auc", 0)
            trial.report(val_pr, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

    try:
        model.fit(
            Xw_tr, yw_tr,
            validation_data=(Xw_va, yw_va),
            batch_size=batch_size,
            epochs=MAX_EPOCHS,
            class_weight=cw,
            callbacks=[
                callbacks.EarlyStopping(
                    monitor="val_pr_auc", mode="max",
                    patience=EARLY_STOP_PAT, restore_best_weights=True),
                callbacks.ReduceLROnPlateau(
                    monitor="val_pr_auc", mode="max",
                    patience=REDUCE_LR_PAT, factor=0.5, min_lr=1e-6),
                OptunaCallback(),
            ],
            verbose=0
        )
    except optuna.TrialPruned:
        raise

    y_pred = model.predict(Xw_va, verbose=0).ravel()
    val_pr_auc = average_precision_score(yw_va, y_pred)

    del model
    K.clear_session()
    return val_pr_auc


# ── Run Optuna ──────────────────────────────────────────────────────
sampler = TPESampler(seed=SEED)
pruner  = MedianPruner(n_startup_trials=5, n_warmup_steps=5)
study   = optuna.create_study(direction="maximize",
                              sampler=sampler, pruner=pruner)
study.optimize(optuna_objective, n_trials=OPTUNA_TRIALS, show_progress_bar=True)

best_params = study.best_params
print(f"\n═══ Best Optuna Params (PR-AUC = {study.best_value:.4f}) ═══")
for k, v in best_params.items():
    print(f"  {k:20s} = {v}")

with open(os.path.join(OUTPUT_DIR, "best_params.json"), "w") as f:
    json.dump(best_params, f, indent=2)
print(f"✅ Best params saved to {OUTPUT_DIR}/best_params.json")


[I 2026-02-18 14:24:55,344] A new study created in memory with name: no-name-f916b4de-d679-46f2-8fa8-38cd3b114def
  0%|          | 0/30 [00:00<?, ?it/s]2026-02-18 14:24:55.673484: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-02-18 14:25:01.822269: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not

[I 2026-02-18 14:25:46,343] Trial 0 finished with value: 0.9995919090579996 and parameters: {'window_len': 10, 'cnn_filters': 48, 'kernel_size': 3, 'lstm_units': 32, 'dropout': 0.45000000000000007, 'lr': 0.0006358358856676254, 'batch_size': 128, 'l2_reg': 0.0003142880890840109}. Best is trial 0 with value: 0.9995919090579996.


2026-02-18 14:25:54.656931: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-02-18 14:31:57.034368: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

[I 2026-02-18 14:31:58,770] Trial 1 finished with value: 0.9992925163231661 and parameters: {'window_len': 30, 'cnn_filters': 80, 'kernel_size': 5, 'lstm_units': 64, 'dropout': 0.35, 'lr': 2.621087878265438e-05, 'batch_size': 128, 'l2_reg': 0.0002267398652378039}. Best is trial 0 with value: 0.9995919090579996.


2026-02-18 14:32:13.868644: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-02-18 14:37:42.762404: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

...

[I 2026-02-18 15:08:47,527] Trial 27 pruned. 


2026-02-18 15:08:58.026822: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
Best trial: 8. Best value: 0.999837:  97%|█████████▋| 29/30 [44:36<01:24, 84.03s/it] 

[I 2026-02-18 15:09:32,285] Trial 28 pruned. 


2026-02-18 15:09:42.206587: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}
2026-02-18 15:11:36.676991: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),

[I 2026-02-18 15:11:38,670] Trial 29 finished with value: 0.9996715583028095 and parameters: {'window_len': 10, 'cnn_filters': 112, 'kernel_size': 3, 'lstm_units': 64, 'dropout': 0.45000000000000007, 'lr': 0.0008283036412062437, 'batch_size': 32, 'l2_reg': 0.00013994000624242627}. Best is trial 8 with value: 0.9998372151366502.

═══ Best Optuna Params (PR-AUC = 0.9998) ═══
  window_len           = 5
  cnn_filters          = 48
  kernel_size          = 3
  lstm_units           = 64
  dropout              = 0.15000000000000002
  lr                   = 0.006153085601625313
  batch_size           = 128
  l2_reg               = 0.0002576417442523316
✅ Best params saved to outputs_detector/best_params.json


## 8. Final Training & Evaluation

In [20]:
# ═══════════════════════════════════════════════════════════════════
# Section 8 — Final Training & Evaluation
# ═══════════════════════════════════════════════════════════════════
seed_everything()
bp = best_params
BEST_WL = bp["window_len"]

# Re-window with best window_len — Mode A
(Xw_tr_A, yw_tr_A, Xw_va_A, yw_va_A, Xw_te_A, yw_te_A), \
    main_imputer, main_scaler = preprocess_split(
        X_all, y_all, idx_train_A, idx_val_A, idx_test_A, feature_cols,
        window_len=BEST_WL, stride=STRIDE)

# Re-window — Mode B (chronological)
if HAS_CHRONO:
    (Xw_tr_B, yw_tr_B, Xw_va_B, yw_va_B, Xw_te_B, yw_te_B), _, _ = \
        preprocess_split(
            X_all, y_all, idx_train_B, idx_val_B, idx_test_B, feature_cols,
            window_len=BEST_WL, stride=STRIDE,
            fit_new=False, imputer=main_imputer, scaler=main_scaler)

n_features = Xw_tr_A.shape[2]
final_model = build_cnn_lstm(
    BEST_WL, n_features,
    cnn_filters=bp["cnn_filters"],
    kernel_size=bp["kernel_size"],
    lstm_units=bp["lstm_units"],
    dropout_rate=bp["dropout"],
    l2_reg=bp["l2_reg"],
    learning_rate=bp["lr"])

final_history = train_and_evaluate(
    final_model, Xw_tr_A, yw_tr_A, Xw_va_A, yw_va_A,
    batch_size=bp["batch_size"], verbose=1)

plot_training_curves(final_history, prefix="08_final")

# Save model
model_path = os.path.join(OUTPUT_DIR, "model_final.keras")
final_model.save(model_path)
print(f"✅ Model saved: {model_path}")


Epoch 1/100


2026-02-18 15:12:35.475052: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}


49/50 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - auc: 0.8989 - loss: 0.4229 - pr_auc: 0.8794

2026-02-18 15:12:41.117500: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


50/50 ━━━━━━━━━━━━━━━━━━━━ 6s 39ms/step - auc: 0.9805 - loss: 0.2537 - pr_auc: 0.9711 - val_auc: 0.9983 - val_loss: 0.1126 - val_pr_auc: 0.9959 - learning_rate: 0.0062
Epoch 2/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 26ms/step - auc: 0.9956 - loss: 0.1184 - pr_auc: 0.9926 - val_auc: 0.9969 - val_loss: 0.1135 - val_pr_auc: 0.9975 - learning_rate: 0.0062
Epoch 3/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - auc: 0.9981 - loss: 0.0927 - pr_auc: 0.9968 - val_auc: 0.9970 - val_loss: 0.0985 - val_pr_auc: 0.9977 - learning_rate: 0.0062
Epoch 4/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 25ms/step - auc: 0.9981 - loss: 0.0775 - pr_auc: 0.9966 - val_auc: 0.9978 - val_loss: 0.0744 - val_pr_auc: 0.9983 - learning_rate: 0.0062
Epoch 5/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - auc: 0.9984 - loss: 0.0743 - pr_auc: 0.9978 - val_auc: 0.9997 - val_loss: 0.0674 - val_pr_auc: 0.9996 - learning_rate: 0.0062
Epoch 6/100
50/50 ━━━━━━━━━━━━━━━━━━━━ 1s 27ms/step - auc: 0.9985 - loss: 0.0713 - pr_auc: 0.9975 - val_auc: 0.9

In [21]:
# ── Comprehensive metrics helper ────────────────────────────────────
def evaluate_full(model, Xw, yw, split_name="Test"):
    """Compute all required metrics and return dict."""
    y_prob = model.predict(Xw, verbose=0).ravel()
    y_pred = (y_prob >= 0.5).astype(int)

    roc  = roc_auc_score(yw, y_prob)
    pr   = average_precision_score(yw, y_prob)
    f1   = f1_score(yw, y_pred)
    prec = precision_score(yw, y_pred, zero_division=0)
    rec  = recall_score(yw, y_pred, zero_division=0)
    cm   = confusion_matrix(yw, y_pred)
    tn, fp, fn, tp = cm.ravel()
    fpr  = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr  = fn / (fn + tp) if (fn + tp) > 0 else 0

    metrics = {
        "split": split_name,
        "ROC-AUC": roc, "PR-AUC": pr,
        "F1": f1, "Precision": prec, "Recall": rec,
        "FPR": fpr, "FNR": fnr,
        "TP": int(tp), "FP": int(fp), "TN": int(tn), "FN": int(fn),
    }

    print(f"\n── {split_name} Metrics ──")
    for k, v in metrics.items():
        if k == "split": continue
        print(f"  {k:12s}: {v:.4f}" if isinstance(v, float) else f"  {k:12s}: {v}")

    return metrics, y_prob, y_pred, cm


def plot_confusion_matrix(cm, split_name, prefix="08"):
    fig, ax = plt.subplots(figsize=(5, 4))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["Benign", "Attack"],
                yticklabels=["Benign", "Attack"])
    ax.set_xlabel("Predicted"); ax.set_ylabel("Actual")
    ax.set_title(f"Confusion Matrix — {split_name}")
    save_fig(fig, f"{prefix}_confusion_{split_name.lower().replace(' ', '_')}")


def plot_roc_pr(yw, y_prob, split_name, prefix="08"):
    # ROC
    fpr_arr, tpr_arr, _ = roc_curve(yw, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(fpr_arr, tpr_arr, color="darkorange", lw=2,
            label=f"ROC-AUC = {roc_auc_score(yw, y_prob):.4f}")
    ax.plot([0,1],[0,1], "--", color="gray")
    ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
    ax.set_title(f"ROC Curve — {split_name}")
    ax.legend(); ax.grid(True, alpha=0.3)
    save_fig(fig, f"{prefix}_roc_{split_name.lower().replace(' ', '_')}")

    # PR
    prec_arr, rec_arr, _ = precision_recall_curve(yw, y_prob)
    fig, ax = plt.subplots(figsize=(6, 5))
    ax.plot(rec_arr, prec_arr, color="navy", lw=2,
            label=f"PR-AUC = {average_precision_score(yw, y_prob):.4f}")
    ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
    ax.set_title(f"PR Curve — {split_name}")
    ax.legend(); ax.grid(True, alpha=0.3)
    save_fig(fig, f"{prefix}_pr_{split_name.lower().replace(' ', '_')}")


print("✅ Evaluation helpers defined.")


✅ Evaluation helpers defined.


In [22]:
# ── Evaluate on all splits ──────────────────────────────────────────
all_metrics = {}

m, p_val, _, cm_val = evaluate_full(final_model, Xw_va_A, yw_va_A, "Val_A")
all_metrics["Val_A"] = m
plot_confusion_matrix(cm_val, "Val_A")
plot_roc_pr(yw_va_A, p_val, "Val_A")

m, p_te, _, cm_te = evaluate_full(final_model, Xw_te_A, yw_te_A, "Test_A")
all_metrics["Test_A"] = m
plot_confusion_matrix(cm_te, "Test_A")
plot_roc_pr(yw_te_A, p_te, "Test_A")

if HAS_CHRONO:
    m, p_teB, _, cm_teB = evaluate_full(final_model, Xw_te_B, yw_te_B, "Test_B_chrono")
    all_metrics["Test_B_chrono"] = m
    plot_confusion_matrix(cm_teB, "Test_B_chrono")
    plot_roc_pr(yw_te_B, p_teB, "Test_B_chrono")

# ── Detection latency estimate ──────────────────────────────────────
print("\n── Detection Latency Estimate ──")
y_test_labels = yw_te_A
y_test_preds  = (p_te >= 0.5).astype(int)
attack_onsets = np.where(np.diff(y_test_labels.astype(int)) == 1)[0] + 1
delays = []
for onset in attack_onsets:
    subsequent_preds = y_test_preds[onset:]
    detected = np.where(subsequent_preds == 1)[0]
    if len(detected) > 0:
        delays.append(detected[0])
if delays:
    print(f"  Mean detection delay: {np.mean(delays):.1f} steps")
    print(f"  Max detection delay:  {np.max(delays)} steps")
else:
    print("  No attack onset transitions found in test set.")

# Save metrics
with open(os.path.join(OUTPUT_DIR, "metrics.json"), "w") as f:
    json.dump(all_metrics, f, indent=2, default=str)
print(f"\n✅ Metrics saved to {OUTPUT_DIR}/metrics.json")


2026-02-18 15:13:20.955503: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}



── Val_A Metrics ──
  ROC-AUC     : 0.9997
  PR-AUC      : 0.9996
  F1          : 0.9867
  Precision   : 0.9818
  Recall      : 0.9916
  FPR         : 0.0144
  FNR         : 0.0084
  TP          : 592
  FP          : 11
  TN          : 755
  FN          : 5
  ✅ Saved plot: outputs_detector/08_confusion_val_a.png
  ✅ Saved plot: outputs_detector/08_roc_val_a.png
  ✅ Saved plot: outputs_detector/08_pr_val_a.png

── Test_A Metrics ──
  ROC-AUC     : 0.9988
  PR-AUC      : 0.9984
  F1          : 0.9801
  Precision   : 0.9704
  Recall      : 0.9899
  FPR         : 0.0235
  FNR         : 0.0101
  TP          : 591
  FP          : 18
  TN          : 748
  FN          : 6
  ✅ Saved plot: outputs_detector/08_confusion_test_a.png
  ✅ Saved plot: outputs_detector/08_roc_test_a.png
  ✅ Saved plot: outputs_detector/08_pr_test_a.png

── Test_B_chrono Metrics ──
  ROC-AUC     : 0.9969
  PR-AUC      : 0.9978
  F1          : 0.9685
  Precision   : 0.9565
  Recall      : 0.9809
  FPR         : 0.0603
 

## 9. Calibration

Compute calibration metrics, apply temperature scaling, export optimal threshold.

In [23]:
# ═══════════════════════════════════════════════════════════════════
# Section 9 — Calibration
# ═══════════════════════════════════════════════════════════════════

# ── 9a. Reliability diagram + ECE ───────────────────────────────────
def reliability_diagram(y_true, y_prob, n_bins=10, title="Reliability", prefix="09"):
    """Plot reliability diagram and return ECE."""
    bin_edges = np.linspace(0, 1, n_bins + 1)
    bin_means, bin_accs, bin_counts = [], [], []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (y_prob >= lo) & (y_prob < hi)
        if mask.sum() == 0:
            continue
        bin_means.append(y_prob[mask].mean())
        bin_accs.append(y_true[mask].mean())
        bin_counts.append(mask.sum())

    bin_means  = np.array(bin_means)
    bin_accs   = np.array(bin_accs)
    bin_counts = np.array(bin_counts)
    ece = np.sum(bin_counts * np.abs(bin_accs - bin_means)) / bin_counts.sum()

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.bar(bin_means, bin_accs, width=0.08, alpha=0.6, edgecolor="black",
           label="Model")
    ax.plot([0,1],[0,1], "--", color="gray", label="Perfectly calibrated")
    ax.set_xlabel("Mean Predicted Probability")
    ax.set_ylabel("Fraction of Positives")
    ax.set_title(f"{title}  (ECE = {ece:.4f})")
    ax.legend(); ax.grid(True, alpha=0.3)
    save_fig(fig, f"{prefix}_reliability_{title.lower().replace(' ', '_')}")
    return ece

# Uncalibrated
p_val_uncal = final_model.predict(Xw_va_A, verbose=0).ravel()
ece_uncal = reliability_diagram(yw_va_A, p_val_uncal,
                                title="Uncalibrated", prefix="09")
print(f"Uncalibrated ECE: {ece_uncal:.4f}")
brier_uncal = brier_score_loss(yw_va_A, p_val_uncal)
print(f"Uncalibrated Brier: {brier_uncal:.4f}")


2026-02-18 15:14:17.129028: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


  ✅ Saved plot: outputs_detector/09_reliability_uncalibrated.png
Uncalibrated ECE: 0.0091
Uncalibrated Brier: 0.0089


In [24]:
# ── 9b. Temperature scaling ────────────────────────────────────────
from scipy.optimize import minimize_scalar

def temperature_scale(logits, T):
    return 1.0 / (1.0 + np.exp(-logits / T))

# Convert probabilities to logits
eps = 1e-7
logits_val = np.log(np.clip(p_val_uncal, eps, 1-eps) /
                    np.clip(1 - p_val_uncal, eps, 1-eps))

def nll_loss(T):
    probs = temperature_scale(logits_val, T)
    return -np.mean(yw_va_A * np.log(probs + eps) +
                    (1 - yw_va_A) * np.log(1 - probs + eps))

result = minimize_scalar(nll_loss, bounds=(0.1, 10.0), method="bounded")
T_opt = result.x
print(f"Optimal temperature: {T_opt:.4f}")

# Apply to val
p_val_cal = temperature_scale(logits_val, T_opt)
ece_cal = reliability_diagram(yw_va_A, p_val_cal,
                               title="Calibrated", prefix="09")
brier_cal = brier_score_loss(yw_va_A, p_val_cal)
print(f"Calibrated   ECE:   {ece_cal:.4f}")
print(f"Calibrated   Brier: {brier_cal:.4f}")

# Apply to test
p_te_uncal = p_te  # from section 8
logits_te = np.log(np.clip(p_te_uncal, eps, 1-eps) /
                   np.clip(1 - p_te_uncal, eps, 1-eps))
p_te_cal = temperature_scale(logits_te, T_opt)

print(f"\nCalibrated vs Uncalibrated (test):")
print(f"  ROC-AUC  uncal={roc_auc_score(yw_te_A, p_te_uncal):.4f}  "
      f"cal={roc_auc_score(yw_te_A, p_te_cal):.4f}")
print(f"  PR-AUC   uncal={average_precision_score(yw_te_A, p_te_uncal):.4f}  "
      f"cal={average_precision_score(yw_te_A, p_te_cal):.4f}")

# Save temperature
calib_info = {"temperature": T_opt,
              "ECE_uncalibrated": float(ece_uncal),
              "ECE_calibrated": float(ece_cal),
              "brier_uncalibrated": float(brier_uncal),
              "brier_calibrated": float(brier_cal)}
with open(os.path.join(OUTPUT_DIR, "calibration.json"), "w") as f:
    json.dump(calib_info, f, indent=2)
print(f"\n✅ Calibration info saved.")


Optimal temperature: 0.9568
  ✅ Saved plot: outputs_detector/09_reliability_calibrated.png
Calibrated   ECE:   0.0090
Calibrated   Brier: 0.0090

Calibrated vs Uncalibrated (test):
  ROC-AUC  uncal=0.9988  cal=0.9988
  PR-AUC   uncal=0.9984  cal=0.9984

✅ Calibration info saved.


In [25]:
# ── 9c. Threshold selection ─────────────────────────────────────────
thresholds = np.linspace(0.01, 0.99, 200)
f1_scores = [f1_score(yw_va_A, (p_val_cal >= t).astype(int), zero_division=0)
             for t in thresholds]
best_idx = np.argmax(f1_scores)
BEST_THRESHOLD = thresholds[best_idx]
best_f1 = f1_scores[best_idx]
print(f"\nOptimal threshold (F1-maximising): {BEST_THRESHOLD:.4f}  (F1={best_f1:.4f})")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(thresholds, f1_scores, color="teal")
ax.axvline(BEST_THRESHOLD, color="red", ls="--",
           label=f"Best θ = {BEST_THRESHOLD:.3f}")
ax.set_xlabel("Threshold"); ax.set_ylabel("F1 Score")
ax.set_title("F1 vs Threshold (Calibrated Probabilities)")
ax.legend(); ax.grid(True, alpha=0.3)
save_fig(fig, "09_f1_vs_threshold")

# Re-evaluate test with optimal threshold
y_te_opt = (p_te_cal >= BEST_THRESHOLD).astype(int)
print(f"\nTest metrics at optimal threshold {BEST_THRESHOLD:.4f}:")
print(classification_report(yw_te_A, y_te_opt,
                             target_names=["Benign", "Attack"]))



Optimal threshold (F1-maximising): 0.8472  (F1=0.9899)
  ✅ Saved plot: outputs_detector/09_f1_vs_threshold.png

Test metrics at optimal threshold 0.8472:
              precision    recall  f1-score   support

      Benign       0.98      0.99      0.98       766
      Attack       0.98      0.97      0.98       597

    accuracy                           0.98      1363
   macro avg       0.98      0.98      0.98      1363
weighted avg       0.98      0.98      0.98      1363



## 10. Cross-Validation Stability (5-Fold Stratified)

In [26]:
# ═══════════════════════════════════════════════════════════════════
# Section 10 — Cross-Validation Stability
# ═══════════════════════════════════════════════════════════════════
# CV uses train+val only (no test leakage).

bp = best_params  # from Optuna
BEST_WL = bp["window_len"]
idx_cv_pool = np.concatenate([idx_train_A, idx_val_A])
y_cv_pool   = y_all[idx_cv_pool]

skf = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=SEED)
cv_results = []

for fold, (tr_rel, va_rel) in enumerate(skf.split(idx_cv_pool, y_cv_pool)):
    seed_everything()
    print(f"\n── Fold {fold+1}/{CV_FOLDS} ──")
    idx_tr_fold = idx_cv_pool[tr_rel]
    idx_va_fold = idx_cv_pool[va_rel]

    (Xw_tr_f, yw_tr_f, Xw_va_f, yw_va_f, _, _), _, _ = preprocess_split(
        X_all, y_all, idx_tr_fold, idx_va_fold,
        idx_va_fold,  # dummy test
        feature_cols, window_len=BEST_WL, stride=STRIDE)

    nf = Xw_tr_f.shape[2]
    m = build_cnn_lstm(BEST_WL, nf,
                       cnn_filters=bp["cnn_filters"],
                       kernel_size=bp["kernel_size"],
                       lstm_units=bp["lstm_units"],
                       dropout_rate=bp["dropout"],
                       l2_reg=bp["l2_reg"],
                       learning_rate=bp["lr"])
    cw = compute_class_weights(yw_tr_f)
    m.fit(Xw_tr_f, yw_tr_f,
          validation_data=(Xw_va_f, yw_va_f),
          batch_size=bp["batch_size"],
          epochs=MAX_EPOCHS,
          class_weight=cw,
          callbacks=get_callbacks(),
          verbose=0)

    y_prob_f = m.predict(Xw_va_f, verbose=0).ravel()
    y_pred_f = (y_prob_f >= BEST_THRESHOLD).astype(int)
    cv_results.append({
        "fold": fold + 1,
        "ROC-AUC": roc_auc_score(yw_va_f, y_prob_f),
        "PR-AUC":  average_precision_score(yw_va_f, y_prob_f),
        "F1":      f1_score(yw_va_f, y_pred_f),
        "Precision": precision_score(yw_va_f, y_pred_f, zero_division=0),
        "Recall":    recall_score(yw_va_f, y_pred_f, zero_division=0),
    })
    print(f"  ROC-AUC={cv_results[-1]['ROC-AUC']:.4f}  "
          f"PR-AUC={cv_results[-1]['PR-AUC']:.4f}  "
          f"F1={cv_results[-1]['F1']:.4f}")
    del m
    K.clear_session()

cv_df = pd.DataFrame(cv_results)
print("\n── CV Summary ──")
for col in ["ROC-AUC", "PR-AUC", "F1", "Precision", "Recall"]:
    print(f"  {col:12s}: {cv_df[col].mean():.4f} ± {cv_df[col].std():.4f}")

cv_df.to_csv(os.path.join(OUTPUT_DIR, "cv_metrics.csv"), index=False)
print(f"✅ CV metrics saved to {OUTPUT_DIR}/cv_metrics.csv")

# Boxplot
fig, ax = plt.subplots(figsize=(8, 5))
cv_df[["ROC-AUC", "PR-AUC", "F1", "Precision", "Recall"]].boxplot(ax=ax)
ax.set_title(f"{CV_FOLDS}-Fold Cross-Validation Metric Distributions")
ax.set_ylabel("Score")
ax.grid(True, alpha=0.3)
save_fig(fig, "10_cv_boxplot")



── Fold 1/5 ──


2026-02-18 15:18:58.892571: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_16}}
2026-02-18 15:19:04.451658: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),


Epoch 7: ReduceLROnPlateau reducing learning rate to 0.003076542867347598.
Epoch 9: early stopping
Restoring model weights from the end of the best epoch: 4.


2026-02-18 15:19:15.847735: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


  ROC-AUC=0.9995  PR-AUC=0.9994  F1=0.9813

── Fold 2/5 ──


...

2026-02-18 15:20:14.777409: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}



Epoch 6: ReduceLROnPlateau reducing learning rate to 0.003076542867347598.
Epoch 8: early stopping
Restoring model weights from the end of the best epoch: 3.


2026-02-18 15:20:25.003794: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


  ROC-AUC=0.9969  PR-AUC=0.9967  F1=0.9691

── CV Summary ──
  ROC-AUC     : 0.9984 ± 0.0011
  PR-AUC      : 0.9981 ± 0.0013
  F1          : 0.9791 ± 0.0058
  Precision   : 0.9916 ± 0.0055
  Recall      : 0.9670 ± 0.0117
✅ CV metrics saved to outputs_detector/cv_metrics.csv
  ✅ Saved plot: outputs_detector/10_cv_boxplot.png


'outputs_detector/10_cv_boxplot.png'

## 11. PPO-Ready Output Spec

In [27]:
# ═══════════════════════════════════════════════════════════════════
# Section 11 — PPO-Ready Output Spec
# ═══════════════════════════════════════════════════════════════════

def predict_detector_outputs(df_input, model=None, imputer_obj=None,
                              scaler_obj=None, feature_list=None,
                              window_len=None, threshold=None,
                              temperature=None):
    """
    Given a raw (or pre-cleaned) DataFrame, return a DataFrame with:
      - attack_probability  (calibrated 0–1)
      - attack_binary       (thresholded 0/1)
      - severity_score      (0–1)
      - confidence          (0–1)

    Parameters are loaded from saved artifacts if not provided.
    """
    if model is None:
        model = keras.models.load_model(os.path.join(OUTPUT_DIR, "model_final.keras"))
    if imputer_obj is None:
        with open(os.path.join(OUTPUT_DIR, "imputer.pkl"), "rb") as f:
            imputer_obj = pickle.load(f)
    if scaler_obj is None:
        with open(os.path.join(OUTPUT_DIR, "scaler.pkl"), "rb") as f:
            scaler_obj = pickle.load(f)
    if feature_list is None:
        feature_list = feature_cols   # from notebook scope
    if window_len is None:
        window_len = BEST_WL
    if threshold is None:
        threshold = BEST_THRESHOLD
    if temperature is None:
        with open(os.path.join(OUTPUT_DIR, "calibration.json")) as f:
            temperature = json.load(f)["temperature"]

    # Extract features
    X = df_input[feature_list].values
    X = imputer_obj.transform(X)
    X = scaler_obj.transform(X)

    # Window
    Xw, _ = create_sequences(X, np.zeros(len(X)), window_len, stride=1)

    # Predict
    raw_prob = model.predict(Xw, verbose=0).ravel()

    # Calibrate (temperature scaling)
    eps = 1e-7
    logits = np.log(np.clip(raw_prob, eps, 1-eps) /
                    np.clip(1-raw_prob, eps, 1-eps))
    cal_prob = 1.0 / (1.0 + np.exp(-logits / temperature))

    # Severity score
    def safe_norm(series):
        s = series.values if hasattr(series, 'values') else series
        s = np.nan_to_num(s, 0)
        mn, mx = s.min(), s.max()
        return (s - mn) / (mx - mn + 1e-10)

    # For severity, we need original (uncleaned) columns from df_input
    # Use the last row of each window for alignment
    n_out = len(cal_prob)
    pad = len(df_input) - n_out
    idx_end = np.arange(pad, len(df_input))  # aligned to window ends

    syn_ratio_raw = df_input["syn_ratio"].values[idx_end] if "syn_ratio" in df_input.columns else np.zeros(n_out)
    syn_flows_raw = df_input["syn_flows"].values[idx_end] if "syn_flows" in df_input.columns else np.zeros(n_out)
    flow_rate_raw = df_input["flow_arrival_rate"].values[idx_end] if "flow_arrival_rate" in df_input.columns else np.zeros(n_out)
    unique_dsts_raw = df_input["unique_dsts"].values[idx_end] if "unique_dsts" in df_input.columns else np.ones(n_out)

    severity = (0.4 * safe_norm(np.nan_to_num(syn_ratio_raw, 0)) +
                0.3 * safe_norm(np.nan_to_num(syn_flows_raw, 0)) +
                0.2 * safe_norm(np.nan_to_num(flow_rate_raw, 0)) +
                0.1 * (1 - safe_norm(np.nan_to_num(unique_dsts_raw, 0))))

    out = pd.DataFrame({
        "attack_probability": cal_prob,
        "attack_binary":      (cal_prob >= threshold).astype(int),
        "severity_score":     severity,
        "confidence":         np.abs(cal_prob - 0.5) * 2,
    })
    return out

print("✅ predict_detector_outputs() defined.")


✅ predict_detector_outputs() defined.


In [28]:
# ── Generate PPO-ready CSV from full dataset ────────────────────────
ppo_out = predict_detector_outputs(df, model=final_model,
                                    imputer_obj=main_imputer,
                                    scaler_obj=main_scaler,
                                    feature_list=feature_cols,
                                    window_len=BEST_WL,
                                    threshold=BEST_THRESHOLD,
                                    temperature=T_opt)

# Align with original timestamps where possible
if ts_for_split is not None:
    ts_vals = ts_for_split.values[BEST_WL-1:]  # aligned to window ends
    ppo_out.insert(0, "timestamp_unix", ts_vals[:len(ppo_out)])

ppo_path = os.path.join(OUTPUT_DIR, "ppo_detector_outputs.csv")
ppo_out.to_csv(ppo_path, index=False)
print(f"\nPPO-ready outputs: {ppo_out.shape}")
print(ppo_out.describe())
print(f"\n✅ Saved: {ppo_path}")


2026-02-18 15:21:49.415880: E tensorflow/core/framework/node_def_util.cc:680] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}



PPO-ready outputs: (9107, 5)
       timestamp_unix  attack_probability  attack_binary  severity_score  \
count    9.107000e+03         9107.000000    9107.000000    9.107000e+03   
mean     1.769372e+09            0.443854       0.434501    1.412717e-01   
std      1.092339e+06            0.489741       0.495719    1.381338e-01   
min      1.768206e+09            0.000155       0.000000    1.428568e-12   
25%      1.768211e+09            0.001490       0.000000    4.285714e-02   
50%      1.769065e+09            0.005680       0.000000    7.375591e-02   
75%      1.769607e+09            0.999010       1.000000    2.054663e-01   
max      1.771324e+09            0.999963       1.000000    8.463758e-01   

        confidence  
count  9107.000000  
mean      0.982139  
std       0.085408  
min       0.017634  
25%       0.994179  
50%       0.997345  
75%       0.998806  
max       0.999925  

✅ Saved: outputs_detector/ppo_detector_outputs.csv


## 12. Reproducibility & Artifact Saving

In [29]:
# ═══════════════════════════════════════════════════════════════════
# Section 12 — Reproducibility & Artifact Saving
# ═══════════════════════════════════════════════════════════════════

# ── Config JSON ─────────────────────────────────────────────────────
config = {
    "DATA_PATH": DATA_PATH,
    "LABEL_COL": LABEL_COL,
    "SEED": SEED,
    "BEST_WINDOW_LEN": BEST_WL,
    "STRIDE": STRIDE,
    "TEST_FRAC": TEST_FRAC,
    "VAL_FRAC": VAL_FRAC,
    "BEST_THRESHOLD": float(BEST_THRESHOLD),
    "TEMPERATURE": float(T_opt),
    "OPTUNA_TRIALS": OPTUNA_TRIALS,
    "best_params": best_params,
    "n_features": int(n_features),
    "feature_cols": feature_cols,
    "timestamp": datetime.now().isoformat(),
}
with open(os.path.join(OUTPUT_DIR, "config.json"), "w") as f:
    json.dump(config, f, indent=2)

# ── Environment versions ────────────────────────────────────────────
import subprocess
try:
    freeze = subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        stderr=subprocess.DEVNULL).decode()
    with open(os.path.join(OUTPUT_DIR, "requirements.txt"), "w") as f:
        f.write(freeze)
except Exception:
    print("⚠ Could not capture pip freeze.")

# ── Summary of all saved artifacts ──────────────────────────────────
print("\n═══ Saved Artifacts ═══")
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size = os.path.getsize(fpath)
    if size > 1024*1024:
        sz_str = f"{size/1024/1024:.1f} MB"
    elif size > 1024:
        sz_str = f"{size/1024:.1f} KB"
    else:
        sz_str = f"{size} B"
    print(f"  {fname:45s}  {sz_str:>10s}")

print("\n✅ All artifacts saved. Notebook complete!")



═══ Saved Artifacts ═══
  01_label_distribution.png                         30.2 KB
  01_missing_values_bar.png                         76.4 KB
  03_correlation_heatmap.png                       336.4 KB
  06_loss_curves.png                                57.9 KB
  06_pr_auc_curves.png                              60.2 KB
  08_confusion_test_a.png                           43.3 KB
  08_confusion_test_b_chrono.png                    45.1 KB
  08_confusion_val_a.png                            42.1 KB
  08_final_loss_curves.png                          68.0 KB
  08_final_pr_auc_curves.png                        64.0 KB
  08_pr_test_a.png                                  38.5 KB
  08_pr_test_b_chrono.png                           39.8 KB
  08_pr_val_a.png                                   37.8 KB
  08_roc_test_a.png                                 52.2 KB
  08_roc_test_b_chrono.png                          54.1 KB
  08_roc_val_a.png                                  51.3 KB
  09_f1_vs_thre